# DoclingDocument & Chunker Inspector

Tool for poking at the structured artifact that flows from docling parse → docling-graph chunker → LLM extraction. Useful for:

- Confirming a PDF parsed the way you expect (pages, tables, picture descriptions, body element types)
- Finding orphan elements (numbers/text fragments that should have been part of a table)
- Inspecting how `HybridChunker` slices a document into chunks before the LLM sees them
- Demonstrating cross-page table fragmentation bugs (the SA-2 missile spec table is the running example)

**Default loaded document:** the SA-2 Guideline PDF (`38bebd4a-9137-4f02-ab64-ec08c94b804c`). To inspect a different document, change `DOC_ID` in §1.

**Required services:** the API container (`eip-mmdpp-api-1`) must be reachable at `http://localhost:8005` for the `/docling-raw` endpoint.

## §1 Configuration

In [1]:
import json
import urllib.request
from collections import Counter
from pathlib import Path

# API endpoint for fetching parsed DoclingDocument JSON.
# Default = Docker-internal DNS (works when this notebook runs inside the
# eip-mmdpp-jupyter container, which is the standard setup). If you are running
# Jupyter directly on the host, change to http://localhost:8005.
API_BASE = "http://api:8000"

# Which document to inspect. Default = SA-2 Guideline PDF used as the running example
# for the cross-page table fragmentation bug.
DOC_ID = "38bebd4a-9137-4f02-ab64-ec08c94b804c"

# Chunker configuration matching docker/docling-graph/app/config_builder.py defaults.
CHUNK_MAX_TOKENS = 4096
MERGE_PEERS = True

# Optional: dump the loaded JSON to disk for offline poking.
DUMP_PATH = Path("/tmp/docling_doc_inspect.json")


## §2 Load DoclingDocument from the API

Hits `GET /v1/documents/{DOC_ID}/docling-raw`, which returns the parsed-and-enriched DoclingDocument JSON straight from the artifact store. The endpoint adds an `_enrichments` field that isn't part of the DoclingDocument schema — we strip it before validating.

If the parsed JSON is large (>10MB), the request can take 5-30 seconds.

In [2]:
url = f"{API_BASE}/v1/documents/{DOC_ID}/docling-raw"
print(f"Fetching {url} ...")
with urllib.request.urlopen(url, timeout=60) as r:
    raw = r.read()
print(f"Received {len(raw):,} bytes")

doc_json = json.loads(raw)

# Persist for offline use; comment out if undesired.
DUMP_PATH.write_bytes(raw)
print(f"Saved to {DUMP_PATH}")

# Strip the API-added enrichments field before constructing DoclingDocument
# (the model_validate call would reject unknown top-level keys otherwise).
enrichments = doc_json.pop("_enrichments", None)
print(f"_enrichments present: {enrichments is not None} ({type(enrichments).__name__ if enrichments else 'None'})")

Fetching http://api:8000/v1/documents/38bebd4a-9137-4f02-ab64-ec08c94b804c/docling-raw ...
Received 24,457,742 bytes
Saved to /tmp/docling_doc_inspect.json
_enrichments present: True (dict)


In [3]:
# Reconstruct a typed DoclingDocument so we can pass it to the chunker.
from docling_core.types.doc.document import DoclingDocument

doc = DoclingDocument.model_validate(doc_json)
print(f"DoclingDocument: name={doc.name!r}")
print(f"  origin: {doc.origin.filename if doc.origin else 'no-origin'}")
print(f"  pages: {len(doc.pages) if doc.pages else 0}")
print(f"  texts: {len(doc.texts)}")
print(f"  tables: {len(doc.tables)}")
print(f"  pictures: {len(doc.pictures)}")
print(f"  groups: {len(doc.groups) if doc.groups else 0}")

DoclingDocument: name='tmp8_vmc3kn'
  origin: tmp8_vmc3kn.pdf
  pages: 28
  texts: 308
  tables: 2
  pictures: 34
  groups: 6


## §3 Document structure — top-level stats

Element-type counts and per-page distribution. Useful first sniff for whether the doc parsed sensibly.

In [4]:
def page_of(prov):
    """Extract the (1-indexed) page number from a docling Prov entry, or None."""
    if not prov:
        return None
    p = prov[0]
    return getattr(p, "page_no", None) or getattr(p, "page", None)

labels = Counter()
pages_text = Counter()
pages_tables = Counter()
pages_pictures = Counter()

for t in doc.texts:
    labels[str(t.label)] += 1
    p = page_of(t.prov)
    if p is not None:
        pages_text[p] += 1

for tab in doc.tables:
    p = page_of(tab.prov)
    if p is not None:
        pages_tables[p] += 1

for pic in doc.pictures:
    p = page_of(pic.prov)
    if p is not None:
        pages_pictures[p] += 1

print("Text element labels (label -> count):")
for lbl, cnt in labels.most_common():
    print(f"  {lbl}: {cnt}")

print("\nTables on each page:")
for p, c in sorted(pages_tables.items()):
    print(f"  page {p}: {c}")

print("\nPages with the most text elements (top 10):")
for p, c in pages_text.most_common(10):
    print(f"  page {p}: {c} text elements")

Text element labels (label -> count):
  text: 109
  page_header: 60
  page_footer: 56
  list_item: 34
  caption: 28
  section_header: 20
  code: 1

Tables on each page:
  page 6: 1
  page 7: 1

Pages with the most text elements (top 10):
  page 1: 30 text elements
  page 3: 29 text elements
  page 27: 29 text elements
  page 2: 24 text elements
  page 8: 22 text elements
  page 7: 19 text elements
  page 5: 16 text elements
  page 4: 14 text elements
  page 6: 11 text elements
  page 14: 9 text elements


## §4 Tables — render cell content

Walks each table, prints page number and a markdown render of cells.

In [5]:
def render_table_markdown(table) -> str:
    """Lightweight markdown render: walk table.data.grid (or .table_cells), emit pipe-separated rows.

    Falls back to `str(table)` if the structure is unexpected.
    """
    data = getattr(table, "data", None)
    if data is None:
        return f"<no .data on {type(table).__name__}>"
    grid = getattr(data, "grid", None)
    if grid:
        # grid is list[list[TableCell]]
        out = []
        for row in grid:
            cells = [getattr(c, "text", "") or "" for c in row]
            out.append("| " + " | ".join(cells) + " |")
        return "\n".join(out)
    cells = getattr(data, "table_cells", None) or []
    if cells:
        # Project cells onto a (max_row+1) x (max_col+1) grid.
        max_r = max(c.start_row_offset_idx for c in cells)
        max_c = max(c.start_col_offset_idx for c in cells)
        mat = [[""] * (max_c + 1) for _ in range(max_r + 1)]
        for c in cells:
            txt = getattr(c, "text", "") or ""
            mat[c.start_row_offset_idx][c.start_col_offset_idx] = txt
        return "\n".join("| " + " | ".join(row) + " |" for row in mat)
    return f"<no grid or table_cells on {type(data).__name__}>"

for i, tab in enumerate(doc.tables):
    p = page_of(tab.prov)
    md = render_table_markdown(tab)
    print(f"=== TABLE {i} (page {p}) — {md.count(chr(10))+1} rows ===")
    # Truncate long renders
    if len(md) > 2500:
        print(md[:2500] + "\n... [truncated]")
    else:
        print(md)
    print()

=== TABLE 0 (page 6) — 22 rows ===
| Industry Designation | Industry Designation | SA-75 | S-75 | S-75M |  |  | S-75V | S-75V |  |  | S-75M |
| Military Designation | Military Designation | SA-75 | S-75 | S-75 | S-75M1 | S-75M1 | S-75M | S-75M | S-75M2 | S-75M4 | S-75 |
| NATO Designation | NATO Designation | SA-2A | SA-2C | SA-2D | SA-2D | SA-2D | SA-2C | SA-2C | SA-2D | SA-2D | SA-2E |
| Fan Song Variant | Fan Song Variant | RSNA- 75 | RSN-75 | RSN- 75M | RSN- 75V1 | RSN- 75V1 | RSN- 75V | RSN- 75V | RSNA- 75M | RSN- 75M4 | RSN- 75M |
| Max Range | m | 29000 | 34000 | 43000 | 34000 | 43000 | 43000 | 45000 | 56000 | 76000 |  |
| Min Range | m | 8000 | 8000 | 8000 |  | 7000 | 7000 | 7000 | 6000 | 6000 |  |
| Max Alt | m | 22000 | 27000 | 30000 | 27000 | 30000 | 30000 | 30000 | 30000 | 30000 |  |
| Min Alt | m | 3000 | 3000 | 1000 | 500 | 300 | 1000 | 1000 | 100 | 50 | 5000 |
| Vmax appr tgt | m/s | 417 | 417 | 639 | 556 | 639 | 639 |  | 1000 | 1000 |  |
| Vmax reced tgt | m/s |  |  |  

## §5 Page boundaries — find orphan elements

When docling splits a multi-page table, the continuation rows often parse as **standalone text elements** rather than table cells. They look like loose numbers/labels with no surrounding structure.

Pages with an unusually high count of short numeric/label text elements are a signal that a table got fragmented. For the SA-2 doc, look at page 7 — that's where the lost "2nd Stage Weight" row lives as orphan text fragments (`1028`, `1251`, `1257`, ...).

In [6]:
# Helper: short numeric-looking text elements (suspect orphan table cells).
import re
NUMBER_LIKE = re.compile(r"^[\d,\.\s]+$")

def is_orphan_candidate(text: str) -> bool:
    s = (text or "").strip()
    if not s or len(s) > 30:
        return False
    return bool(NUMBER_LIKE.match(s))

# Per-page: count orphan candidates, flag pages with notable runs of them.
page_orphans = Counter()
page_orphan_samples: dict[int, list[str]] = {}
for t in doc.texts:
    if is_orphan_candidate(t.text or t.orig or ""):
        p = page_of(t.prov)
        if p is None:
            continue
        page_orphans[p] += 1
        page_orphan_samples.setdefault(p, []).append((t.text or t.orig or "").strip())

print("Pages with the most orphan-candidate text elements (top 8):")
for p, cnt in page_orphans.most_common(8):
    samples = page_orphan_samples.get(p, [])[:8]
    print(f"  page {p}: {cnt} orphans — sample: {samples}")

Pages with the most orphan-candidate text elements (top 8):
  page 7: 9 orphans — sample: ['1257', '1380', '1380', '1386', '1028', '1251', '1251', '1257']
  page 3: 1 orphans — sample: ['15']
  page 4: 1 orphans — sample: ['11964']
  page 8: 1 orphans — sample: ['50']
  page 14: 1 orphans — sample: ['00']


In [7]:
# Specific check for SA-2: are the expected-but-missing sustain values (1028, 1251, 1257, 1380, 1386, 1399) present as orphan texts?
EXPECTED_SUSTAIN_VALUES = {"1028", "1251", "1257", "1380", "1386", "1399"}
found = []
for t in doc.texts:
    s = (t.text or t.orig or "").strip()
    if s in EXPECTED_SUSTAIN_VALUES:
        found.append((page_of(t.prov), str(t.label), s))

if found:
    print("Found expected sustain values as standalone text elements:")
    for p, lbl, s in found:
        print(f"  page {p}: label={lbl!r} value={s!r}")
    print()
    print("This is the bug fingerprint: numbers that SHOULD be in a table cell have been parsed as plain text.")
else:
    print("No matches found — either the document doesn't have those values, or they're inside a table (not orphaned).")

Found expected sustain values as standalone text elements:
  page 7: label='page_header' value='1257'
  page 7: label='page_header' value='1380'
  page 7: label='page_header' value='1380'
  page 7: label='page_header' value='1386'
  page 7: label='text' value='1028'
  page 7: label='text' value='1251'
  page 7: label='text' value='1251'
  page 7: label='page_header' value='1257'
  page 7: label='text' value='1399'

This is the bug fingerprint: numbers that SHOULD be in a table cell have been parsed as plain text.


## §6 Run the chunker

Uses Docling's `HybridChunker` with the same configuration the docling-graph service uses (`chunk_max_tokens=512`, `merge_peers=True`). The chunker walks the document body in reading order and emits structure-preserving chunks of bounded token count. These chunks are what eventually get stuffed into the LLM prompt.

Note: `HybridChunker` may produce chunks slightly over `chunk_max_tokens` (e.g., a single long table that doesn't have a natural split point). That's by design.

In [8]:
from docling.chunking import HybridChunker

chunker = HybridChunker(
    chunk_max_tokens=CHUNK_MAX_TOKENS,
    merge_peers=MERGE_PEERS,
)

# chunk_iter() returns DocChunk objects (text + meta + doc_items refs).
# We materialize for inspection.
chunks = list(chunker.chunk(doc))
print(f"Chunker produced {len(chunks)} chunks")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (2011 > 512). Running this sequence through the model will result in indexing errors


Chunker produced 125 chunks


In [9]:
# Inspect each chunk: token count (approximate via str length / 4), page span, content preview.
def chunk_pages(chunk) -> list[int]:
    """Return the set of page numbers covered by a chunk's referenced doc_items."""
    pages = set()
    items = getattr(chunk.meta, "doc_items", None) or []
    for item in items:
        prov = getattr(item, "prov", None) or []
        for p in prov:
            pn = getattr(p, "page_no", None) or getattr(p, "page", None)
            if pn is not None:
                pages.add(pn)
    return sorted(pages)

print(f"{'idx':>4} {'pages':>14} {'~chars':>8}  preview")
print("-" * 100)
for i, c in enumerate(chunks):
    text = c.text
    pages = chunk_pages(c)
    page_str = (
        f"{pages[0]}"
        if len(pages) <= 1
        else f"{pages[0]}..{pages[-1]} ({len(pages)})"
    )
    preview = text.replace("\n", " \\n ")
    print(f"{i:>4} {page_str:>14} {len(text):>8}  {preview!r}")

 idx          pages   ~chars  preview
----------------------------------------------------------------------------------------------------
   0              1      153  '[FIFB-22](https://www.ausairpower.net/raptor.html) \\n [PACRIM WEPS](https://www.ausairpower.net/region.html) \\n - [Ready to win bigger; \\n faster and smarter with'
   1              1      401  'AI?](http://d.adroll.com/click/?adroll_insertion_id=48760b031b457241b2fc010a98a6d01c&adroll_pixalate_click_url=https%3A//adrta.com/c%3Fclid%3Dar%26paid%3Dar%26avid%3D4ZYN5F45WFCBFID26NI42R%26caid%3DHGVJGN57U5HLNJREUOP7UN%26plid%3DXIWONOR5PFHSJNNQJNHGXV%26siteId%3Dausairpower.net%26kv1%3D728x90%26publisherId%3Dpub-8664514669849908%26kv2%3Dhttps%253a%252f%252fwww.ausairpower.net%252fAPA-S-75-Volkhov.'
   2              1      196  'html%26kv3%3D1debdccc062ab7af3be05d10d9f6513b%26kv4%3D136.53.88. \\n 0%26kv7%3DBA%26kv10%3D%5BISP%5D%26kv11%3D8310425266134763789430582991558063169%26kv18%3D%26kv19%3D%5BDevice_ID%5D%26kv24%3DDeskto

## §7 Cross-chunk analysis — does any chunk see both Weight rows?

The bug fingerprint for SA-2: the LLM never sees the 1st Stage Weight row and the 2nd Stage Weight row in the same chunk (or in chunks where both are recognizable as table cells). If the chunker had merged the page-6 table with its page-7 continuation, an LLM looking at that chunk would extract both booster_mass_kg AND sustain_mass_kg correctly.

Below we look for the marker phrases and identify which chunk each ends up in.

In [10]:
# For SA-2, the booster Weight row in the parsed table contains "1135" "1032" etc.
# The lost sustain Weight row's values are 1028 1251 1257 1380 1386 1399.
BOOSTER_VALUES = ["1135", "1032", "1011", "1007"]
SUSTAIN_VALUES = ["1028", "1251", "1257", "1380", "1386", "1399"]

def chunks_containing(chunks, needles):
    out = []
    for i, c in enumerate(chunks):
        hits = [n for n in needles if n in c.text]
        if hits:
            out.append((i, hits, chunk_pages(c)))
    return out

booster_chunks = chunks_containing(chunks, BOOSTER_VALUES)
sustain_chunks = chunks_containing(chunks, SUSTAIN_VALUES)

print("Chunks containing booster-row values:")
for i, hits, pages in booster_chunks:
    print(f"  chunk {i:>3} (pages {pages}): hits={hits}")

print("\nChunks containing sustain-row values:")
for i, hits, pages in sustain_chunks:
    print(f"  chunk {i:>3} (pages {pages}): hits={hits}")

overlap = {i for i, _, _ in booster_chunks} & {i for i, _, _ in sustain_chunks}
print()
if overlap:
    print(f"OVERLAP: chunk(s) {overlap} contain BOTH booster and sustain row values — chunker united them.")
else:
    print("NO OVERLAP: booster and sustain row values live in different chunks.")
    print("This is the bug fingerprint — the LLM seeing the booster Weight chunk has no")
    print("line of sight to the sustain Weight values, and the chunker emitting the sustain")
    print("row has lost the 'Weight kg' row label that would let the LLM map the values.")

Chunks containing booster-row values:
  chunk  52 (pages [6]): hits=['1135', '1032', '1011', '1007']

Chunks containing sustain-row values:
  chunk  53 (pages [6, 7]): hits=['1028', '1251']
  chunk  58 (pages [7, 8]): hits=['1399']

NO OVERLAP: booster and sustain row values live in different chunks.
This is the bug fingerprint — the LLM seeing the booster Weight chunk has no
line of sight to the sustain Weight values, and the chunker emitting the sustain
row has lost the 'Weight kg' row label that would let the LLM map the values.


In [11]:
# Show the actual content of the chunk(s) holding the orphan sustain values, so the
# loss of structure is visible.
for i, hits, pages in sustain_chunks:
    print(f"=== CHUNK {i} (pages {pages}) ===")
    print(chunks[i].text[:1500])
    if len(chunks[i].text) > 1500:
        print(f"... [truncated, full length {len(chunks[i].text)}]")
    print()

=== CHUNK 53 (pages [6, 7]) ===
 2nd Stage. 2nd Stage, 10 = 2nd Stage. 2nd Stage, 11 = 2nd Stage. Diameter, 1 = mm. Diameter, 2 = 500. Diameter, 3 = 500. Diameter, 4 = 500. Diameter, 5 = 500. Diameter, 6 = 500. Diameter, 7 = 500. Diameter, 8 = 500. Diameter, 9 = 500. Diameter, 10 = 500. Diameter, 11 = . Span, 1 = mm. Span, 2 = . Span, 3 = 1691. Span, 4 = 1691. Span, 5 = 1691. Span, 6 = 1691. Span, 7 = 1691. Span, 8 = 1691. Span, 9 = 1691. Span, 10 = 1691. Span, 11 = 
kg
1028
1251
1251
Source: http://www.rzeszow.mm.pl/~jowitek/S-75.html / Vestnik PVO

=== CHUNK 58 (pages [7, 8]) ===
1399
This image is classified as an engineering_drawing with high confidence; a block_diagram is a plausible alternate as it represents functional spatial relationships, but the inclusion of a metric scale and site geometry confirms it as a technical site plan. The image is a top-down schematic layout of a "Typical SA-2 Guideline Battalion Launch Site" (Russian: Схема боевой позиции зрди С-75), depicting a c

## §8 HybridChunker configuration harness — extraction pass (Pass B)

Models the **graph-extraction chunker** (the pass that feeds chunk text into the LLM for graph extraction). Does NOT model the embedding pass.

**Production extraction pass uses:** `sentence-transformers/all-MiniLM-L6-v2` tokenizer (hardcoded in docling-graph's `DocumentChunker`), `max_tokens=4096` (from `DOCLING_GRAPH_CHUNK_MAX_TOKENS`), output goes to `gemma4:31b` for graph extraction.

The harness fixes the tokenizer to MiniLM and sweeps every other HybridChunker knob.

- §8.1 — `ChunkerConfig` dataclass and `build_chunker` helper.
- §8.2 — Cartesian sweep (5 max_tokens × 2⁴ booleans = 80 configs).
- §8.3 — sweep all configs; print summary table.
- §8.4 — per-config drill-down (full text of every chunk).
- §8.5 — chunk-level diff between any two configs.
- §8.6 — Stage-3-split detection: smallest `max_tokens` that keeps every source item whole.
- §8.7 — per-knob illustrative examples (one cell per knob, side-by-side).

> Note: the chunker's tokenizer is a *proxy* for the LLM's tokenization. MiniLM and Gemma's SentencePiece will count slightly differently. For modeling production exactly, MiniLM is correct; for deciding what `max_tokens` to actually use, the LLM's real context window is the binding constraint.
>
> Note: §6's `HybridChunker(chunk_max_tokens=...)` call uses a deprecated parameter — it only works via a backcompat `@model_validator` shim in upstream Docling. The harness below uses the correct path: `max_tokens` is set on the tokenizer, then the tokenizer is passed to `HybridChunker`.


In [12]:
# §8.1 — ChunkerConfig dataclass + build_chunker helper.
#
# Every HybridChunker knob is exposed here. The tokenizer is built from
# tokenizer_kind + tokenizer_model + max_tokens. tiktoken is guarded so
# the cell still works if the optional extra is not installed.

from dataclasses import dataclass
from typing import Literal

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

# Counting cap mirrors docling-graph (document_chunker.py): keep
# tokenizer.model_max_length above max_tokens so encoding does not
# silently truncate when chunks brush the limit.
_TOKENIZER_COUNTING_MAX_LENGTH = 8192


@dataclass(frozen=True)
class ChunkerConfig:
    name: str
    tokenizer_kind: Literal["hf", "tiktoken"] = "hf"
    tokenizer_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    max_tokens: int = 4096
    merge_peers: bool = True
    always_emit_headings: bool = False
    repeat_table_header: bool = True
    omit_header_on_overflow: bool = False


def _build_tokenizer(config: ChunkerConfig):
    """Build the tokenizer for a config. Returns None if tiktoken is requested
    but the extra is not installed (caller skips the config gracefully)."""
    if config.tokenizer_kind == "hf":
        from transformers import AutoTokenizer

        hf = AutoTokenizer.from_pretrained(config.tokenizer_model)
        new_max = max(config.max_tokens, _TOKENIZER_COUNTING_MAX_LENGTH)
        if hf.model_max_length < new_max:
            hf.model_max_length = new_max
        return HuggingFaceTokenizer(tokenizer=hf, max_tokens=config.max_tokens)

    if config.tokenizer_kind == "tiktoken":
        try:
            import tiktoken
            from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
        except ImportError:
            return None
        # tokenizer_model for tiktoken == encoding name (e.g. "cl100k_base").
        tt = tiktoken.get_encoding(config.tokenizer_model)
        return OpenAITokenizer(tokenizer=tt, max_tokens=config.max_tokens)

    raise ValueError(f"Unknown tokenizer_kind: {config.tokenizer_kind!r}")


def build_chunker(config: ChunkerConfig) -> HybridChunker | None:
    """Build a HybridChunker from a config. Returns None if a required
    dependency (e.g. tiktoken) is unavailable."""
    tok = _build_tokenizer(config)
    if tok is None:
        return None
    return HybridChunker(
        tokenizer=tok,
        merge_peers=config.merge_peers,
        always_emit_headings=config.always_emit_headings,
        repeat_table_header=config.repeat_table_header,
        omit_header_on_overflow=config.omit_header_on_overflow,
    )


In [13]:
# §8.2 — Configs to compare.
#
# Tokenizer fixed to sentence-transformers/all-MiniLM-L6-v2 — matches the
# live extraction pass (Pass B) in docling-graph's DocumentChunker.
#
# Naming scheme: mt<max_tokens>_mp<0/1>_hd<0/1>_rh<0/1>_oo<0/1>
#   mt = max_tokens
#   mp = merge_peers
#   hd = always_emit_headings
#   rh = repeat_table_header
#   oo = omit_header_on_overflow
#
# Live extraction pass ≡ mt4096_mp1_hd0_rh1_oo0
#   (max_tokens=4096, merge_peers=True, always_emit_headings=False,
#    repeat_table_header=True, omit_header_on_overflow=False)
#
# Edit MAX_TOKENS_VALUES below to shrink/grow the size sweep.
# Default 5 sizes × 2⁴ boolean combos = 80 configs.

from itertools import product

TOKENIZER = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS_VALUES = [512, 1024, 2048, 4096, 8192]
BOOL_VALUES = [True, False]

CONFIGS: list[ChunkerConfig] = []
for max_tokens, merge, headings, repeat_hdr, omit_overflow in product(
    MAX_TOKENS_VALUES, BOOL_VALUES, BOOL_VALUES, BOOL_VALUES, BOOL_VALUES
):
    name = (
        f"mt{max_tokens}"
        f"_mp{int(merge)}"
        f"_hd{int(headings)}"
        f"_rh{int(repeat_hdr)}"
        f"_oo{int(omit_overflow)}"
    )
    CONFIGS.append(ChunkerConfig(
        name=name,
        tokenizer_model=TOKENIZER,
        max_tokens=max_tokens,
        merge_peers=merge,
        always_emit_headings=headings,
        repeat_table_header=repeat_hdr,
        omit_header_on_overflow=omit_overflow,
    ))

assert len({c.name for c in CONFIGS}) == len(CONFIGS), "config names must be unique"

print(f"Defined {len(CONFIGS)} configs (tokenizer fixed to {TOKENIZER!r}).")
print(f"Sweep: max_tokens ∈ {MAX_TOKENS_VALUES}  ×  2⁴ boolean combinations")
print(f"Live-pipeline equivalent: 'mt4096_mp1_hd0_rh1_oo0'  (production extraction pass)")
print()
print("First few + last few:")
for c in CONFIGS[:3] + CONFIGS[-3:]:
    print(
        f"  • {c.name:<28} max_tokens={c.max_tokens:<5} mp={c.merge_peers!s:<5} "
        f"hd={c.always_emit_headings!s:<5} rh={c.repeat_table_header!s:<5} "
        f"oo={c.omit_header_on_overflow!s}"
    )


Defined 80 configs (tokenizer fixed to 'sentence-transformers/all-MiniLM-L6-v2').
Sweep: max_tokens ∈ [512, 1024, 2048, 4096, 8192]  ×  2⁴ boolean combinations
Live-pipeline equivalent: 'mt4096_mp1_hd0_rh1_oo0'  (production extraction pass)

First few + last few:
  • mt512_mp1_hd1_rh1_oo1        max_tokens=512   mp=True  hd=True  rh=True  oo=True
  • mt512_mp1_hd1_rh1_oo0        max_tokens=512   mp=True  hd=True  rh=True  oo=False
  • mt512_mp1_hd1_rh0_oo1        max_tokens=512   mp=True  hd=True  rh=False oo=True
  • mt8192_mp0_hd0_rh1_oo0       max_tokens=8192  mp=False hd=False rh=True  oo=False
  • mt8192_mp0_hd0_rh0_oo1       max_tokens=8192  mp=False hd=False rh=False oo=True
  • mt8192_mp0_hd0_rh0_oo0       max_tokens=8192  mp=False hd=False rh=False oo=False


In [14]:
# §8.3 — Sweep every config; print summary table.
# Caches chunks in chunks_by_config and token counts in token_counts_by_config
# for §8.4 / §8.5 / §8.6 / §8.7.

import statistics


def _chunk_pages(chunk) -> list[int]:
    pages = set()
    for item in getattr(chunk.meta, "doc_items", None) or []:
        for p in getattr(item, "prov", None) or []:
            pn = getattr(p, "page_no", None) or getattr(p, "page", None)
            if pn is not None:
                pages.add(pn)
    return sorted(pages)


def _chunk_self_refs(chunk) -> list[str]:
    refs = []
    for item in getattr(chunk.meta, "doc_items", None) or []:
        ref = getattr(item, "self_ref", None)
        if isinstance(ref, str) and ref:
            refs.append(ref)
    return refs


def _has_table(chunk) -> bool:
    for item in getattr(chunk.meta, "doc_items", None) or []:
        ref = getattr(item, "self_ref", "") or ""
        if "/tables/" in ref:
            return True
    return False


def _is_heading_only(chunk) -> bool:
    items = getattr(chunk.meta, "doc_items", None) or []
    if items:
        return False
    body = (chunk.text or "").strip()
    if not body:
        return True
    headings = getattr(chunk.meta, "headings", None) or []
    return body == "\n".join(headings).strip()


chunks_by_config: dict[str, list] = {}
chunkers_by_config: dict[str, HybridChunker] = {}
token_counts_by_config: dict[str, list[int]] = {}
summary_rows: list[dict] = []

for config in CONFIGS:
    chunker = build_chunker(config)
    if chunker is None:
        print(f"[skip] {config.name}: dependency unavailable")
        continue

    chunks = list(chunker.chunk(doc))
    chunks_by_config[config.name] = chunks
    chunkers_by_config[config.name] = chunker

    tok = chunker.tokenizer
    token_counts = [tok.count_tokens(chunker.contextualize(chunk=c)) for c in chunks]
    token_counts_by_config[config.name] = token_counts

    summary_rows.append({
        "name": config.name,
        "n_chunks": len(chunks),
        "tok_min": min(token_counts) if token_counts else 0,
        "tok_med": int(statistics.median(token_counts)) if token_counts else 0,
        "tok_max": max(token_counts) if token_counts else 0,
        "page_cross": sum(1 for c in chunks if len(_chunk_pages(c)) > 1),
        "tables": sum(1 for c in chunks if _has_table(c)),
        "headonly": sum(1 for c in chunks if _is_heading_only(c)),
    })

print()
header = f"{'config':<28} {'#chunks':>8} {'tok_min':>8} {'tok_med':>8} {'tok_max':>8} {'pg_cross':>9} {'tables':>7} {'hdonly':>7}"
print(header)
print("-" * len(header))
for r in summary_rows:
    print(
        f"{r['name']:<28} {r['n_chunks']:>8} {r['tok_min']:>8} {r['tok_med']:>8} {r['tok_max']:>8} "
        f"{r['page_cross']:>9} {r['tables']:>7} {r['headonly']:>7}"
    )

print()
print("Legend:  pg_cross = chunks spanning >1 page,  tables = chunks containing ≥1 table,  hdonly = chunks with only headings, no body items")



config                        #chunks  tok_min  tok_med  tok_max  pg_cross  tables  hdonly
------------------------------------------------------------------------------------------
mt512_mp1_hd1_rh1_oo1              60        5      422      519         6       6       0
mt512_mp1_hd1_rh1_oo0              60        5      422      519         6       6       0
mt512_mp1_hd1_rh0_oo1              60        5      422      512         6       6       0
mt512_mp1_hd1_rh0_oo0              60        5      422      512         6       6       0
mt512_mp1_hd0_rh1_oo1              58        7      429      519         6       6       0
mt512_mp1_hd0_rh1_oo0              58        7      429      519         6       6       0
mt512_mp1_hd0_rh0_oo1              58        7      429      512         6       6       0
mt512_mp1_hd0_rh0_oo0              58        7      429      512         6       6       0
mt512_mp0_hd1_rh1_oo1             139        2       89      519         2       6       

In [15]:
# §8.4 — Per-config drill-down: full text of every chunk for INSPECT_CONFIG.

# Default = the config that matches the live extraction pass.
INSPECT_CONFIG = "mt4096_mp1_hd0_rh1_oo0"

if INSPECT_CONFIG not in chunks_by_config:
    raise KeyError(
        f"{INSPECT_CONFIG!r} not in chunks_by_config; available: "
        f"{sorted(chunks_by_config)[:10]}... ({len(chunks_by_config)} total)"
    )

chunks = chunks_by_config[INSPECT_CONFIG]
chunker = chunkers_by_config[INSPECT_CONFIG]
tok = chunker.tokenizer

print(f"Drill-down: {INSPECT_CONFIG} — {len(chunks)} chunks")
print("=" * 100)

for i, c in enumerate(chunks):
    pages = _chunk_pages(c)
    if not pages:
        page_str = "—"
    elif len(pages) == 1:
        page_str = f"{pages[0]}"
    else:
        page_str = f"{pages[0]}..{pages[-1]} ({len(pages)} pages)"

    n_tok = tok.count_tokens(chunker.contextualize(chunk=c))
    n_items = len(getattr(c.meta, "doc_items", None) or [])
    headings = getattr(c.meta, "headings", None) or []

    print(f"\n--- chunk {i:>3}  pages={page_str}  tokens={n_tok}  doc_items={n_items} ---")
    if headings:
        print(f"headings: {' › '.join(headings)}")
    print(c.text)


Drill-down: mt4096_mp1_hd0_rh1_oo0 — 19 chunks

--- chunk   0  pages=1  tokens=2058  doc_items=3 ---
[FIFB-22](https://www.ausairpower.net/raptor.html)
[PACRIM WEPS](https://www.ausairpower.net/region.html)
- [Ready to win bigger; faster and smarter with AI?](http://d.adroll.com/click/?adroll_insertion_id=48760b031b457241b2fc010a98a6d01c&adroll_pixalate_click_url=https%3A//adrta.com/c%3Fclid%3Dar%26paid%3Dar%26avid%3D4ZYN5F45WFCBFID26NI42R%26caid%3DHGVJGN57U5HLNJREUOP7UN%26plid%3DXIWONOR5PFHSJNNQJNHGXV%26siteId%3Dausairpower.net%26kv1%3D728x90%26publisherId%3Dpub-8664514669849908%26kv2%3Dhttps%253a%252f%252fwww.ausairpower.net%252fAPA-S-75-Volkhov.html%26kv3%3D1debdccc062ab7af3be05d10d9f6513b%26kv4%3D136.53.88.0%26kv7%3DBA%26kv10%3D%5BISP%5D%26kv11%3D8310425266134763789430582991558063169%26kv18%3D%26kv19%3D%5BDevice_ID%5D%26kv24%3DDesktop&adroll_ad_payload=__HIA9QBkwHFA8HIA70AAZ1TXYjcVBSeZNb6UFHclQF9WlBkHzbZ5Oa_Wkp2dudvd5OZHeeXpfEmuTMTZzLJziQ7uoK0qA9CW-hDEbYoIn2xL2Lx3bp7YbVNhj74okgfRFR

In [16]:
# §8.5 — Diff two configs by self_ref overlap.
#
# For each A-chunk, finds the set of B-chunks that share any of A's
# doc-item self_refs. Pattern reveals splits (A:1 ↔ B:[3,4,5]),
# merges (A:[1,2] ↔ B:7), and 1:1 matches.
#
# Default diff: production extraction (4096) vs. 8192 — quantifies how
# many items get un-fragmented when you double the cap.

CONFIG_A = "mt4096_mp1_hd0_rh1_oo0"
CONFIG_B = "mt8192_mp1_hd0_rh1_oo0"

for name in (CONFIG_A, CONFIG_B):
    if name not in chunks_by_config:
        raise KeyError(
            f"{name!r} not in chunks_by_config; "
            f"available: {sorted(chunks_by_config)[:10]}... "
            f"({len(chunks_by_config)} total)"
        )

a_chunks = chunks_by_config[CONFIG_A]
b_chunks = chunks_by_config[CONFIG_B]

ref_to_b: dict[str, list[int]] = {}
for j, c in enumerate(b_chunks):
    for ref in _chunk_self_refs(c):
        ref_to_b.setdefault(ref, []).append(j)

print(f"Diff: {CONFIG_A} (A, n={len(a_chunks)})  ↔  {CONFIG_B} (B, n={len(b_chunks)})")
print("=" * 100)
print(f"{'A':>4}  {'A_pages':>10}  {'A_refs':>7}  →  B chunks (shared self_refs)")
print("-" * 100)

split_count = match_count = orphan_count = 0
b_to_a: dict[int, list[int]] = {}

for i, ac in enumerate(a_chunks):
    a_refs = _chunk_self_refs(ac)
    a_pages = _chunk_pages(ac)
    b_idxs: set[int] = set()
    for ref in a_refs:
        b_idxs.update(ref_to_b.get(ref, []))
    b_sorted = sorted(b_idxs)

    for j in b_sorted:
        b_to_a.setdefault(j, []).append(i)

    if not b_sorted:
        kind = "ORPHAN"
        orphan_count += 1
    elif len(b_sorted) == 1:
        kind = "1:1"
        match_count += 1
    else:
        kind = f"SPLIT→{len(b_sorted)}"
        split_count += 1

    if len(a_pages) > 1:
        a_page_str = f"{a_pages[0]}..{a_pages[-1]}"
    elif a_pages:
        a_page_str = str(a_pages[0])
    else:
        a_page_str = "—"

    print(f"{i:>4}  {a_page_str:>10}  {len(a_refs):>7}  →  {b_sorted}  [{kind}]")

merges = {j: a_list for j, a_list in b_to_a.items() if len(a_list) > 1}

print()
print(f"Summary: matches(1:1)={match_count}  splits(A→multiple B)={split_count}  orphans(A→none)={orphan_count}")
if merges:
    print(f"Merges (B-chunk ← multiple A-chunks): {len(merges)}")
    for j, a_list in sorted(merges.items()):
        print(f"  B:{j}  ←  A:{a_list}")
else:
    print("Merges: 0")


Diff: mt4096_mp1_hd0_rh1_oo0 (A, n=19)  ↔  mt8192_mp1_hd0_rh1_oo0 (B, n=19)
   A     A_pages   A_refs  →  B chunks (shared self_refs)
----------------------------------------------------------------------------------------------------
   0           1        3  →  [0]  [1:1]
   1           1       11  →  [1]  [1:1]
   2        1..2       28  →  [2]  [1:1]
   3        2..3        4  →  [3]  [1:1]
   4        3..4       28  →  [4]  [1:1]
   5        4..6       17  →  [5]  [1:1]
   6           6        3  →  [6]  [1:1]
   7        6..7        6  →  [7]  [1:1]
   8           7        1  →  [8]  [1:1]
   9       7..10        9  →  [9]  [1:1]
  10      11..13       10  →  [10]  [1:1]
  11      13..14        5  →  [11]  [1:1]
  12      15..16        6  →  [12]  [1:1]
  13          17        2  →  [13]  [1:1]
  14          18        2  →  [14]  [1:1]
  15          19        2  →  [15]  [1:1]
  16      20..27       17  →  [16]  [1:1]
  17          27        2  →  [17]  [1:1]
  18      27..28   

In [17]:
# §8.6 — Find the smallest max_tokens that keeps every source item whole.
#
# Detects HybridChunker Stage-3 splits by counting source items (self_refs)
# that appear in more than one chunk. self_ref appearing N>1 times means
# that source item was fragmented by semchunk — the failure mode that
# produces mid-row table cuts in graph-extraction input.
#
# Reports per config:
#   n_chunks    — total chunks
#   max_tok     — largest single chunk's token count (config's own tokenizer)
#   near_cap    — chunks at ≥90% of max_tokens (under tokenization pressure)
#   n_split     — count of source items that got Stage-3-split (0 = no fragmentation)
#   worst       — largest fragmentation count for any single item (1 = no splits)
#
# Best config for extraction = smallest max_tokens with n_split == 0.

from collections import Counter

print(f"{'config':<28} {'n_chunks':>9} {'max_tok':>8} {'near_cap':>9} {'n_split':>8} {'worst':>6}")
print("-" * 80)

clean_configs: list[str] = []

for config in CONFIGS:
    if config.name not in chunks_by_config:
        continue
    chunks = chunks_by_config[config.name]
    token_counts = token_counts_by_config[config.name]

    max_tok = max(token_counts) if token_counts else 0
    near_cap = sum(1 for t in token_counts if t >= 0.9 * config.max_tokens)

    ref_counter: Counter = Counter()
    for c in chunks:
        for ref in _chunk_self_refs(c):
            ref_counter[ref] += 1
    split_refs = {r: n for r, n in ref_counter.items() if n > 1}
    n_split = len(split_refs)
    worst_split = max(split_refs.values()) if split_refs else 1

    flag = "  ← all items whole" if n_split == 0 else ""
    if n_split == 0:
        clean_configs.append(config.name)

    print(
        f"{config.name:<28} {len(chunks):>9} {max_tok:>8} {near_cap:>9} "
        f"{n_split:>8} {worst_split:>6}{flag}"
    )

print()
if clean_configs:
    clean_by_size: dict[int, list[str]] = {}
    for name in clean_configs:
        cfg = next(c for c in CONFIGS if c.name == name)
        clean_by_size.setdefault(cfg.max_tokens, []).append(name)
    smallest_clean = min(clean_by_size)
    print(f"Smallest max_tokens with zero Stage-3 splits: {smallest_clean}")
    prod_match = f"mt{smallest_clean}_mp1_hd0_rh1_oo0"
    if prod_match in clean_by_size[smallest_clean]:
        print(f"  Recommended: {prod_match}  (matches production knobs at this max_tokens)")
    else:
        print(f"  At max_tokens={smallest_clean}: {clean_by_size[smallest_clean]}")
else:
    largest_swept = max(c.max_tokens for c in CONFIGS)
    print("⚠ NO config in the sweep produces zero Stage-3 splits for this document.")
    print(f"  This document contains at least one source item larger than the largest")
    print(f"  max_tokens swept ({largest_swept}). Increase MAX_TOKENS_VALUES in §8.2,")
    print("  OR consider a custom serializer / pre-splitting large tables.")

print()
print("Items split in the live-extraction config:")
live_name = "mt4096_mp1_hd0_rh1_oo0"
if live_name in chunks_by_config:
    ref_counter = Counter()
    for c in chunks_by_config[live_name]:
        for ref in _chunk_self_refs(c):
            ref_counter[ref] += 1
    splits = sorted(((r, n) for r, n in ref_counter.items() if n > 1), key=lambda x: -x[1])
    if splits:
        print(f"  {len(splits)} items split in {live_name}:")
        for ref, n in splits[:10]:
            print(f"    {ref}  →  {n} fragments")
        if len(splits) > 10:
            print(f"    ... and {len(splits) - 10} more")
    else:
        print(f"  {live_name}: no Stage-3 splits ✓")


config                        n_chunks  max_tok  near_cap  n_split  worst
--------------------------------------------------------------------------------
mt512_mp1_hd1_rh1_oo1               60      519        16        3     12
mt512_mp1_hd1_rh1_oo0               60      519        16        3     12
mt512_mp1_hd1_rh0_oo1               60      512        17        3     12
mt512_mp1_hd1_rh0_oo0               60      512        17        3     12
mt512_mp1_hd0_rh1_oo1               58      519        16        3     12
mt512_mp1_hd0_rh1_oo0               58      519        16        3     12
mt512_mp1_hd0_rh0_oo1               58      512        17        3     12
mt512_mp1_hd0_rh0_oo0               58      512        17        3     12
mt512_mp0_hd1_rh1_oo1              139      519        11        3     12
mt512_mp0_hd1_rh1_oo0              139      519        11        3     12
mt512_mp0_hd1_rh0_oo1              139      512        11        3     12
mt512_mp0_hd1_rh0_oo0          

## §8.7 Per-knob illustrative examples

For each HybridChunker knob, finds and prints a concrete example from this document showing what it does. Uses the swept configs from §8.2 — no extra chunker runs needed.

Each cell:
1. Identifies a baseline chunk and the corresponding chunk(s) in a variant where only that one knob differs.
2. Prints both side-by-side so the effect is readable.
3. Falls back to a clear "no example found in this document" message if the knob's effect didn't surface here.

All examples use `mt4096_mp1_hd0_rh1_oo0` (live extraction config) as the baseline; the variant flips one knob.


In [18]:
# §8.7.1 — max_tokens
# Doubling max_tokens lets the chunker either avoid Stage-3 splits or
# include more peer items via Stage-4 merge. Compares mt512 vs mt4096.

SMALL = "mt512_mp1_hd0_rh1_oo0"
LARGE = "mt4096_mp1_hd0_rh1_oo0"

s_chunks = chunks_by_config[SMALL]
l_chunks = chunks_by_config[LARGE]

print(f"== max_tokens: {SMALL}  vs  {LARGE} ==")
print(f"Chunk count:  {SMALL}={len(s_chunks)}  vs  {LARGE}={len(l_chunks)}  "
      f"(Δ = {len(s_chunks) - len(l_chunks)} fewer at higher cap)")
print(f"Max chunk tokens: {max(token_counts_by_config[SMALL])} vs {max(token_counts_by_config[LARGE])}")
print()

ref_to_large: dict[str, list[int]] = {}
for j, c in enumerate(l_chunks):
    for ref in _chunk_self_refs(c):
        ref_to_large.setdefault(ref, []).append(j)

# Find a SMALL chunk whose refs collapse into a single LARGE chunk that has more refs
# (i.e. SMALL was a fragment / unmerged peer of LARGE).
for i, sc in enumerate(s_chunks):
    s_refs = _chunk_self_refs(sc)
    if not s_refs:
        continue
    targets = set()
    for ref in s_refs:
        targets.update(ref_to_large.get(ref, []))
    if len(targets) == 1:
        j = next(iter(targets))
        l_refs = _chunk_self_refs(l_chunks[j])
        if len(l_refs) > len(s_refs):
            print(f"Example: {SMALL} chunk {i} (n_refs={len(s_refs)}) is part of "
                  f"{LARGE} chunk {j} (n_refs={len(l_refs)})")
            print("-" * 100)
            print(f"[{SMALL}] chunk {i}:")
            t = s_chunks[i].text
            print(t[:600] + (f"  ...[+{len(t) - 600} chars]" if len(t) > 600 else ""))
            print()
            print(f"[{LARGE}] chunk {j}:")
            t = l_chunks[j].text
            print(t[:1200] + (f"  ...[+{len(t) - 1200} chars]" if len(t) > 1200 else ""))
            break
else:
    print(f"No clear max_tokens example found between {SMALL} and {LARGE}.")


== max_tokens: mt512_mp1_hd0_rh1_oo0  vs  mt4096_mp1_hd0_rh1_oo0 ==
Chunk count:  mt512_mp1_hd0_rh1_oo0=58  vs  mt4096_mp1_hd0_rh1_oo0=19  (Δ = 39 fewer at higher cap)
Max chunk tokens: 519 vs 3984

Example: mt512_mp1_hd0_rh1_oo0 chunk 1 (n_refs=1) is part of mt4096_mp1_hd0_rh1_oo0 chunk 0 (n_refs=3)
----------------------------------------------------------------------------------------------------
[mt512_mp1_hd0_rh1_oo0] chunk 1:
adroll_ad_payload=__HIA9QBkwHFA8HIA70AAZ1TXYjcVBSeZNb6UFHclQF9WlBkHzbZ5Oa_Wkp2dudvd5OZHeeXpfEmuTMTZzLJziQ7uoK0qA9CW-hDEbYoIn2xL2Lx3bp7YbVNhj74okgfRFR89UHwB8yohaI-FC_c78DHuZzz3e-cuymVxBmpraulcqml1wuAy6pVvbza0kEaZ3Rpna82RG2jXnhhe6NZqzT0bFOYw5lmsaFr-rZQzhWqJU2rlLRCvlmfx2wvCPzxqZWVyWRCw3AMnZHvTdCIHqJgRS2rVJWSBKruDfo9b4_uBe7gSfzV7E3yZBz6jkdbnnsmDFxj7IUjC52G9sgbDJ6dMS6yndC9n7Gg60OnO7zHQdsKTnfV2Vn1hLzow6BZrrf2s1stymcaepvT7Krjl4bFdmGrPsgVh72uKis1pzcs8DUjy2U1zaq7lVaxHxRe6WuyuZWvQkHcb45t1xTWbNivdUaM7qlsHvG1l_a6W15NkAzY6HR2Q2AwpR4AOVbmN9sbpjCprD2FM3y7pQk5Xmjksqu54hoQtS

In [19]:
# §8.7.2 — merge_peers
# When True, adjacent chunks sharing heading context that fit together
# under max_tokens are merged into one. Compares mt4096_mp1 vs mt4096_mp0.

WITH_MERGE = "mt4096_mp1_hd0_rh1_oo0"
NO_MERGE   = "mt4096_mp0_hd0_rh1_oo0"

m_chunks = chunks_by_config[WITH_MERGE]
n_chunks = chunks_by_config[NO_MERGE]

print(f"== merge_peers: {WITH_MERGE}  vs  {NO_MERGE} ==")
print(f"Chunk count:  with_merge={len(m_chunks)}  vs  no_merge={len(n_chunks)}  "
      f"(Δ = {len(n_chunks) - len(m_chunks)} merged away)")
print()

ref_to_n: dict[str, list[int]] = {}
for j, c in enumerate(n_chunks):
    for ref in _chunk_self_refs(c):
        ref_to_n.setdefault(ref, []).append(j)

# Find a merged chunk: WITH_MERGE chunk whose refs span ≥2 NO_MERGE chunks.
for i, mc in enumerate(m_chunks):
    m_refs = _chunk_self_refs(mc)
    if len(m_refs) < 2:
        continue
    n_targets: set[int] = set()
    for ref in m_refs:
        n_targets.update(ref_to_n.get(ref, []))
    if len(n_targets) >= 2:
        n_sorted = sorted(n_targets)
        print(f"Example: {WITH_MERGE} chunk {i} (n_refs={len(m_refs)}) "
              f"corresponds to {NO_MERGE} chunks {n_sorted}")
        print("-" * 100)
        print(f"[{WITH_MERGE}] chunk {i} (merged):")
        t = mc.text
        print(t[:1000] + (f"  ...[+{len(t) - 1000} chars]" if len(t) > 1000 else ""))
        print()
        for j in n_sorted[:3]:
            print(f"[{NO_MERGE}] chunk {j}:")
            t = n_chunks[j].text
            print(t[:500] + (f"  ...[+{len(t) - 500} chars]" if len(t) > 500 else ""))
            print()
        break
else:
    print("No merge example found in this document — every chunk in WITH_MERGE has 0 or 1 refs,")
    print("or refs never spanned ≥2 chunks in NO_MERGE. merge_peers had no visible effect here.")


== merge_peers: mt4096_mp1_hd0_rh1_oo0  vs  mt4096_mp0_hd0_rh1_oo0 ==
Chunk count:  with_merge=19  vs  no_merge=122  (Δ = 103 merged away)

Example: mt4096_mp1_hd0_rh1_oo0 chunk 0 (n_refs=3) corresponds to mt4096_mp0_hd0_rh1_oo0 chunks [0, 1, 2]
----------------------------------------------------------------------------------------------------
[mt4096_mp1_hd0_rh1_oo0] chunk 0 (merged):
[FIFB-22](https://www.ausairpower.net/raptor.html)
[PACRIM WEPS](https://www.ausairpower.net/region.html)
- [Ready to win bigger; faster and smarter with AI?](http://d.adroll.com/click/?adroll_insertion_id=48760b031b457241b2fc010a98a6d01c&adroll_pixalate_click_url=https%3A//adrta.com/c%3Fclid%3Dar%26paid%3Dar%26avid%3D4ZYN5F45WFCBFID26NI42R%26caid%3DHGVJGN57U5HLNJREUOP7UN%26plid%3DXIWONOR5PFHSJNNQJNHGXV%26siteId%3Dausairpower.net%26kv1%3D728x90%26publisherId%3Dpub-8664514669849908%26kv2%3Dhttps%253a%252f%252fwww.ausairpower.net%252fAPA-S-75-Volkhov.html%26kv3%3D1debdccc062ab7af3be05d10d9f6513b%26kv4%3D1

In [20]:
# §8.7.3 — always_emit_headings
# When True, sections with no body content emit a heading-only chunk that
# would otherwise be silently skipped.

WITHOUT = "mt4096_mp1_hd0_rh1_oo0"
WITH    = "mt4096_mp1_hd1_rh1_oo0"

w_chunks = chunks_by_config[WITHOUT]
h_chunks = chunks_by_config[WITH]

print(f"== always_emit_headings: {WITHOUT}  vs  {WITH} ==")
print(f"Chunk count:  off={len(w_chunks)}  vs  on={len(h_chunks)}  "
      f"(Δ = {len(h_chunks) - len(w_chunks)} extra heading-only chunks)")
print()

heading_only = [(i, c) for i, c in enumerate(h_chunks) if _is_heading_only(c)]
print(f"Heading-only chunks in WITH config: {len(heading_only)}")
print()

if heading_only:
    print(f"Examples (first {min(3, len(heading_only))} heading-only chunks):")
    print("-" * 100)
    for i, c in heading_only[:3]:
        headings = getattr(c.meta, "headings", None) or []
        print(f"[{WITH}] chunk {i}")
        print(f"  headings: {' › '.join(headings) if headings else '(none)'}")
        print(f"  text:     {c.text!r}")
        print()
else:
    print("No empty-section headings in this document — always_emit_headings had no visible effect.")
    print("Headings are only emitted as standalone chunks when a section has 0 body items.")


== always_emit_headings: mt4096_mp1_hd0_rh1_oo0  vs  mt4096_mp1_hd1_rh1_oo0 ==
Chunk count:  off=19  vs  on=21  (Δ = 2 extra heading-only chunks)

Heading-only chunks in WITH config: 0

No empty-section headings in this document — always_emit_headings had no visible effect.
Headings are only emitted as standalone chunks when a section has 0 body items.


In [21]:
# §8.7.4 — repeat_table_header
# When True, a table split across multiple chunks repeats its header in
# each fragment. Only matters when a table is actually split — needs a
# table > max_tokens.

WITH_REPEAT = "mt4096_mp1_hd0_rh1_oo0"
NO_REPEAT   = "mt4096_mp1_hd0_rh0_oo0"

w_chunks = chunks_by_config[WITH_REPEAT]
n_chunks = chunks_by_config[NO_REPEAT]

print(f"== repeat_table_header: {WITH_REPEAT}  vs  {NO_REPEAT} ==")
print(f"Chunk count:  with_repeat={len(w_chunks)}  vs  no_repeat={len(n_chunks)}")
print()


def _find_split_table(chunks):
    counter = Counter()
    for c in chunks:
        for ref in _chunk_self_refs(c):
            if "/tables/" in ref:
                counter[ref] += 1
    return next(((r, n) for r, n in counter.most_common() if n > 1), None)


split_with = _find_split_table(w_chunks)
split_no = _find_split_table(n_chunks)

if split_with is None and split_no is None:
    print("No split tables in this document at mt4096 — repeat_table_header had no visible effect.")
    print("To surface it: lower max_tokens (e.g. switch to a 'mt512' config) so a table gets split,")
    print("then compare mt512_mp1_hd0_rh1_oo0 vs mt512_mp1_hd0_rh0_oo0.")
else:
    target_ref = (split_with or split_no)[0]
    w_idxs = [i for i, c in enumerate(w_chunks) if target_ref in _chunk_self_refs(c)]
    n_idxs = [i for i, c in enumerate(n_chunks) if target_ref in _chunk_self_refs(c)]
    print(f"Split table: self_ref={target_ref}")
    print(f"  with_repeat: appears in {len(w_idxs)} chunks ({w_idxs})")
    print(f"  no_repeat:   appears in {len(n_idxs)} chunks ({n_idxs})")
    print()
    if len(w_idxs) >= 2 and len(n_idxs) >= 2:
        print(f"--- {WITH_REPEAT} fragment 2 (chunk {w_idxs[1]}) ---")
        t = w_chunks[w_idxs[1]].text
        print(t[:600] + (f"  ...[+{len(t) - 600} chars]" if len(t) > 600 else ""))
        print()
        print(f"--- {NO_REPEAT} fragment 2 (chunk {n_idxs[1]}) ---")
        t = n_chunks[n_idxs[1]].text
        print(t[:600] + (f"  ...[+{len(t) - 600} chars]" if len(t) > 600 else ""))
        print()
        print("Look for the table header row at the top of the WITH_REPEAT fragment;")
        print("it should be absent in NO_REPEAT (rows continue mid-table without context).")


== repeat_table_header: mt4096_mp1_hd0_rh1_oo0  vs  mt4096_mp1_hd0_rh0_oo0 ==
Chunk count:  with_repeat=19  vs  no_repeat=19

No split tables in this document at mt4096 — repeat_table_header had no visible effect.
To surface it: lower max_tokens (e.g. switch to a 'mt512' config) so a table gets split,
then compare mt512_mp1_hd0_rh1_oo0 vs mt512_mp1_hd0_rh0_oo0.


In [22]:
# §8.7.5 — omit_header_on_overflow
# When True, the table header is dropped from a chunk if including it
# would push the chunk over max_tokens. Only matters when a row (without
# header) fits, but row + header doesn't.

OFF = "mt4096_mp1_hd0_rh1_oo0"
ON  = "mt4096_mp1_hd0_rh1_oo1"

off_chunks = chunks_by_config[OFF]
on_chunks  = chunks_by_config[ON]

print(f"== omit_header_on_overflow: {OFF}  vs  {ON} ==")
print(f"Chunk count:  off={len(off_chunks)}  vs  on={len(on_chunks)}  "
      f"(Δ = {len(off_chunks) - len(on_chunks)})")
print()

# Compare per-table chunk counts; differing counts ⇒ omit changed splitting.
def _table_chunk_map(chunks):
    out: dict[str, list[int]] = {}
    for i, c in enumerate(chunks):
        for ref in _chunk_self_refs(c):
            if "/tables/" in ref:
                out.setdefault(ref, []).append(i)
    return out


off_map = _table_chunk_map(off_chunks)
on_map  = _table_chunk_map(on_chunks)

diff_refs = [r for r in off_map if len(off_map[r]) != len(on_map.get(r, []))]
if not diff_refs:
    print("No table where omit_header_on_overflow changes chunk count.")
    print("This is normal: omit only matters when (header + row) > max_tokens but row alone fits.")
    print("At mt4096 with the tables in this document, the condition probably never triggers.")
    print("Try lowering max_tokens (e.g. mt512_mp1_hd0_rh1_oo0 vs mt512_mp1_hd0_rh1_oo1) to surface it.")
else:
    target_ref = diff_refs[0]
    off_idxs = off_map[target_ref]
    on_idxs  = on_map.get(target_ref, [])
    print(f"Example: table {target_ref}")
    print(f"  off → {len(off_idxs)} chunks ({off_idxs})")
    print(f"  on  → {len(on_idxs)} chunks ({on_idxs})")
    print()
    print(f"--- {OFF} fragment 1 (chunk {off_idxs[0]}) ---")
    t = off_chunks[off_idxs[0]].text
    print(t[:600] + (f"  ...[+{len(t) - 600} chars]" if len(t) > 600 else ""))
    print()
    print(f"--- {ON} fragment 1 (chunk {on_idxs[0]}) ---")
    t = on_chunks[on_idxs[0]].text
    print(t[:600] + (f"  ...[+{len(t) - 600} chars]" if len(t) > 600 else ""))


== omit_header_on_overflow: mt4096_mp1_hd0_rh1_oo0  vs  mt4096_mp1_hd0_rh1_oo1 ==
Chunk count:  off=19  vs  on=19  (Δ = 0)

No table where omit_header_on_overflow changes chunk count.
This is normal: omit only matters when (header + row) > max_tokens but row alone fits.
At mt4096 with the tables in this document, the condition probably never triggers.
Try lowering max_tokens (e.g. mt512_mp1_hd0_rh1_oo0 vs mt512_mp1_hd0_rh1_oo1) to surface it.


## §9 What you'd do next

Concrete follow-ups depending on what you find:

1. **If the chunker's output looks fine but downstream extraction misses values:** the bug is in `_table_facts.py` / `_alias_map.py` (the parser-side fact extraction layer). Inspect `pipeline_pass_outputs.extract_pass_response_json->'table_overlay'->'facts'` for the same doc and compare expected vs actual schema_field counts.
2. **If a chunk holds both Weight row values but the LLM still can't extract them correctly:** the bug is in the **prompt** or in **schema enforcement** (gemma4:31b at high temperature drops fields). Re-run with `temperature=0.0` or stronger format-mode (`force_json_mode=False` to use schema-grammar mode).
3. **If the Weight rows live in separate chunks AND the row label (`Weight | kg`) only appears in one of them:** the bug is in **docling's table parsing across page boundaries** — the continuation row got demoted to plain text on the next page. Fixing this requires either an upstream docling fix or a custom multi-page table stitcher.

Use this notebook as the starting point for any of those investigations: change `DOC_ID` to a different document and re-run §2-§8.